# Build, release, run

**Objective:** Build code once, combine it with deployment configuration as an immutable release, then run that release without modifying it.

## Simple version

Build creates an artifact, release adds configuration, and run starts the selected release.

In [ ]:
# Each stage has one responsibility and produces input for the next stage.
stages = [
    "build: resolve dependencies and create artifact",
    "release: combine artifact with deployment config",
    "run: start processes from the immutable release",
]

print(*stages, sep="\n")

## Polished version

The same build artifact can produce different environment releases without rebuilding or editing its code.

In [ ]:
import hashlib
from dataclasses import dataclass


# A build artifact identifies code plus locked dependencies, but has no config yet.
@dataclass(frozen=True)
class BuildArtifact:
    commit: str
    digest: str


@dataclass(frozen=True)
class Release:
    version: str
    artifact: BuildArtifact
    config: tuple[tuple[str, str], ...]


def build(commit: str, lockfile: str) -> BuildArtifact:
    # The digest changes whenever a build input changes.
    source = f"{commit}:{lockfile}".encode()
    digest = hashlib.sha256(source).hexdigest()[:12]
    return BuildArtifact(commit=commit, digest=digest)


# A release combines one existing artifact with deployment configuration.
def release(
    artifact: BuildArtifact,
    version: str,
    config: dict[str, str],
) -> Release:
    return Release(
        version=version,
        artifact=artifact,
        config=tuple(sorted(config.items())),
    )


artifact = build(commit="a1b2c3d", lockfile="uv.lock@sha256:abc")
staging = release(artifact, "2026.08.31-rc1", {"APP_ENV": "staging"})
production = release(artifact, "2026.08.31", {"APP_ENV": "production"})

# Promotion reuses the artifact instead of rebuilding it per environment.
assert staging.artifact is production.artifact
print(staging)
print(production)

## Applied in this repository

Project lockfiles and `make sync` define the build inputs. A deployment pipeline should create an immutable release, inject environment config, and start the API or worker from that release.